# 01 — Análisis exploratorio de ventas autorizadas

**Preguntas:** ¿cómo se distribuyen ventas, ceros y decimales? ¿Qué diferencias descriptivas existen entre tiendas/familias? ¿Cómo cambia el volumen mensual y qué cobertura/extremos requieren atención?

Solo se consulta PostgreSQL `training`, lote bootstrap y fechas 2013–2014. No se abre SOURCE, Silver, CSV ni el perfil global de 2015–2017. La conexión está fijada al rol trainer y las consultas son de solo lectura y acotadas. No se cambia el corte. Las ventas son cantidades en unidades/peso heterogéneos, **no ingresos monetarios ni inventario**. Los outputs se limpian antes de conservar los notebooks; los PNG contienen solo agregados.


In [ ]:
from pathlib import Path
import sys
NOTEBOOKS = Path.cwd() if (Path.cwd() / "retailops_eda").is_dir() else Path.cwd() / "notebooks"
sys.path.insert(0, str(NOTEBOOKS))
import pandas as pd
import matplotlib.pyplot as plt
from retailops_eda.data import query, validate_bootstrap
from retailops_eda.analysis import setup, finish, calendar_series, monthly_means, lag_correlations, save_summary, UNIT
setup()
overview = validate_bootstrap()  # trainer, solo lectura, corte fijo y prueba de ausencia futura
print("Bootstrap autorizado: 2013-01-01 → 2014-12-31; filas:", int(overview['records']))


## Distribución y calidad numérica

Los intervalos se fijan de antemano; los cuantiles se calculan exclusivamente sobre el bootstrap visible. Una venta decimal es válida. Cero observado no equivale a ausencia, cierre o demanda perdida. La transformación logarítmica se usa solo para visualizar, no para entrenar.


In [ ]:
display(overview.to_frame("Valor"))
distribution = query("distribution")
labels = ["Cero", "(0, 1)", "[1, 10)", "[10, 100)", "[100, 1.000)", "[1.000, 10.000)", "≥10.000"]
freq = distribution.set_index('bucket')['records'].reindex(range(7), fill_value=0)
fig, ax = plt.subplots()
ax.bar(labels, freq / 1e6, color="#315EAF")
ax.set(title="Distribución de ventas diarias por tienda/familia · 2013–2014", xlabel="Ventas registradas por observación (intervalos de escala creciente)", ylabel="Observaciones (millones)")
ax.tick_params(axis='x', labelrotation=20)
finish(fig, "eda_distribution"); plt.show()
zero_rate = float(overview.zero_sales / overview.records)
fractional_rate = float(overview.decimal_sales / overview.records)
print(f"Ceros: {zero_rate:.2%}; valores no enteros: {fractional_rate:.2%}.")
print("Interpretación: comparar mediana y media para describir asimetría; los valores altos no se eliminan automáticamente.")


## Diferencias entre tiendas y familias

Se usan identificadores de los propios hechos, sin acceder a la dimensión stores ni atribuir ciudad, tipo o aperturas desconocidas. Los volúmenes agregados entre familias no son comparaciones homogéneas de dinero o unidades físicas. Las diferencias describen el período; no explican su causa.


In [ ]:
stores = query('by_store')
families = query('by_family')
fig, ax = plt.subplots(figsize=(13,5))
ax.bar(stores.store_nbr.astype(str), stores.total_sales / 1e6, color="#315EAF")
ax.set(title="Volumen registrado por tienda · 2013–2014", xlabel="Número de tienda", ylabel="Ventas registradas (millones; unidades mixtas)")
for i, label in enumerate(ax.get_xticklabels()): label.set_visible(i % 3 == 0)
finish(fig, 'eda_stores'); plt.show()
fig, ax = plt.subplots(figsize=(12,10))
rank = families.sort_values('total_sales')
ax.barh(rank.family, rank.total_sales / 1e6, color="#237A64")
ax.set(title="Volumen registrado por familia · 2013–2014", xlabel="Ventas registradas (millones; unidades mixtas)", ylabel="Familia")
finish(fig, 'eda_families'); plt.show()
display(stores.assign(zero_rate=stores.zero_sales / stores.records).sort_values('total_sales', ascending=False).head(5))
display(families.assign(zero_rate=families.zero_sales / families.records).head(5))


## Evolución mensual y cobertura

Mostrar el total y la media por día observado evita atribuir al patrón comercial diferencias debidas únicamente a la longitud de los meses. Las dos Navidades ausentes se mantienen como NaN; no se rellenan con cero. El registro de huecos es accesible a través de training.observation_gaps.


In [ ]:
monthly = monthly_means(query('monthly'))
daily = query('daily')
series = calendar_series(daily)
gaps = query('gaps')
expected_grid = int(overview.stores * overview.families)
if not daily.records.eq(expected_grid).all(): raise ValueError('Cuadrícula diaria incompleta')
fig, axes = plt.subplots(2,1,figsize=(12,7),sharex=True)
axes[0].plot(monthly.month, monthly.total_sales / 1e6, marker='o', color='#315EAF')
axes[0].set(title='Evolución mensual · 2013–2014', ylabel='Ventas mensuales (millones)')
axes[1].plot(monthly.month, monthly.sales_per_observed_day / 1e3, marker='o', color='#237A64')
axes[1].set(xlabel='Mes', ylabel='Media por día observado (miles)')
finish(fig, 'eda_monthly'); plt.show()
display(gaps)
print(f"Días observados: {len(daily)}; días sin observación: {int(series.isna().sum())}; claves por día observado: {expected_grid}.")
print('Interpretación: integridad de días presentes y completitud del calendario son conceptos distintos.')


## Extremos y límites de interpretación

Se agregan las observaciones ≥ P99 del propio bootstrap por tienda/familia (máximo 20 grupos), sin exportar filas individuales. P99 es un umbral descriptivo, no una regla de corrupción. Sin existencias, precios ni metadatos disponibles no puede atribuirse un pico a promociones, aperturas o eventos.


In [ ]:
extremes = query('extremes', threshold=float(overview.p99))
display(extremes)
fig, ax = plt.subplots(figsize=(12,5))
ax.scatter(extremes.records, extremes.maximum, color='#9A4D14', alpha=.8)
ax.set(title='Grupos con mayores máximos entre observaciones ≥ P99 · bootstrap', xlabel='Observaciones del grupo por encima del umbral', ylabel='Máximo registrado (unidades/peso mixtos)')
finish(fig, 'eda_extremes'); plt.show()
availability = query('availability')
display(availability)
print('Si una vista auxiliar no devuelve filas, se documenta y se mantiene la restricción; no se consulta Silver.')
save_summary('eda', {'records':int(overview.records), 'total_sales':str(overview.total_sales), 'minimum':str(overview.minimum), 'maximum':str(overview.maximum), 'mean':str(overview.mean_sales), 'p25':float(overview.p25),'p50':float(overview.p50),'p75':float(overview.p75),'p95':float(overview.p95),'p99':float(overview.p99), 'zero_count':int(overview.zero_sales),'fractional_count':int(overview.decimal_sales), 'zero_rate':zero_rate,'fractional_rate':fractional_rate,'stores':int(overview.stores),'families':int(overview.families),'observed_days':len(daily),'missing_dates':[d.date().isoformat() for d in series.index[series.isna()]],'top_store':int(stores.loc[stores.total_sales.idxmax(),'store_nbr']),'top_store_share':float(stores.total_sales.max()/stores.total_sales.sum()),'top_family':str(families.iloc[0].family),'top_family_share':float(families.iloc[0].total_sales/families.total_sales.sum()),'auxiliary_visible':dict(zip(availability.entity,availability.records.astype(int)))})


## Conclusiones y continuación

Las tablas/gráficos anteriores se recalculan al ejecutar el notebook; las cifras verificadas de la sesión se documentan en `docs/EDA_FINDINGS.md`. La asimetría, concentración, ceros y diferencias por familia sugieren comparar baselines y métricas segmentadas, sin retirar extremos automáticamente. No se ha creado GOLD, ajustado un modelo ni utilizado observaciones futuras.
